# GenAI Notebook: Retrieval-Augmented Financial Assistant

This notebook builds a lightweight RAG pipeline for financial questions. It can use Gemini embeddings and generation when credentials are present, and falls back to deterministic local components when they are not.

In [1]:
from pathlib import Path
import json
import os
import re

import joblib
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

try:
    import faiss  # type: ignore
except Exception:
    faiss = None

try:
    import google.generativeai as genai
except Exception:
    genai = None

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / 'backend').exists() and (REPO_ROOT.parent / 'backend').exists():
    REPO_ROOT = REPO_ROOT.parent

RAW_DIR = REPO_ROOT / 'data' / 'raw'
MODEL_DIR = REPO_ROOT / 'backend' / 'models' / 'genai'
RAG_DIR = REPO_ROOT / 'ml_pipeline' / 'models' / 'genai'
RAW_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
RAG_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
np.random.seed(SEED)

f:\project\NEURALALPHA\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def clean_text(text: str) -> str:
    text = re.sub(r'\s+', ' ', str(text)).strip()
    return text

def chunk_text(text: str, chunk_size: int = 500, overlap: int = 80):
    tokens = text.split()
    chunks = []
    start = 0
    while start < len(tokens):
        end = min(len(tokens), start + chunk_size)
        chunks.append(' '.join(tokens[start:end]))
        if end == len(tokens):
            break
        start = max(0, end - overlap)
    return chunks

sample_documents = [
    {
        'source': 'market_regime',
        'text': 'Risk-on regimes generally favor equities, while defensive positioning emphasizes cash flow quality, lower drawdown, and balance sheet strength.'
    },
    {
        'source': 'portfolio_construction',
        'text': 'A diversified portfolio should balance expected return, volatility, and correlation across sectors, factors, and capitalizations.'
    },
    {
        'source': 'earnings_quality',
        'text': 'Sustainable earnings are supported by margin expansion, recurring revenue, prudent leverage, and positive operating cash flow.'
    },
    {
        'source': 'sentiment_signal',
        'text': 'Sentiment should be used as a context feature rather than a sole signal. It is most useful when combined with price momentum and liquidity measures.'
    },
]

corpus_path = RAW_DIR / 'genai_corpus.json'
if corpus_path.exists():
    documents = json.loads(corpus_path.read_text(encoding='utf-8'))
else:
    documents = sample_documents
    corpus_path.write_text(json.dumps(documents, indent=2), encoding='utf-8')

chunks = []
for document in documents:
    source = document.get('source', 'unknown')
    text = clean_text(document.get('text', ''))
    for index, chunk in enumerate(chunk_text(text, chunk_size=80, overlap=12)):
        chunks.append({'source': source, 'chunk_id': index, 'text': chunk})

chunks_df = pd.DataFrame(chunks)
display(chunks_df.head())
print('chunk count:', len(chunks_df))

,source,chunk_id,text
0,market_regime,0,"Risk-on regimes generally favor equities, whil..."
1,portfolio_construction,0,A diversified portfolio should balance expecte...
2,earnings_quality,0,Sustainable earnings are supported by margin e...
3,sentiment_signal,0,Sentiment should be used as a context feature ...


chunk count: 4


In [3]:
embedding_mode = 'tfidf'
vectorizer = TfidfVectorizer(ngram_range=(1, 2), stop_words='english', max_features=3000)
tfidf_matrix = vectorizer.fit_transform(chunks_df['text'])
embeddings = tfidf_matrix.toarray().astype('float32')
embedding_dim = embeddings.shape[1]

if genai is not None and os.getenv('GOOGLE_API_KEY'):
    try:
        genai.configure(api_key=os.getenv('GOOGLE_API_KEY'))
        embedding_mode = 'gemini'
    except Exception:
        embedding_mode = 'tfidf'

print({'embedding_mode': embedding_mode, 'embedding_dim': embedding_dim})

if faiss is not None:
    index = faiss.IndexFlatIP(embedding_dim)
    normalized = embeddings / np.clip(np.linalg.norm(embeddings, axis=1, keepdims=True), 1e-8, None)
    index.add(normalized)
else:
    index = NearestNeighbors(metric='cosine')
    index.fit(embeddings)

def embed_query(query: str):
    vector = vectorizer.transform([clean_text(query)]).toarray().astype('float32')
    return vector

{'embedding_mode': 'tfidf', 'embedding_dim': 94}


In [4]:
def retrieve(query: str, top_k: int = 3):
    query_vector = embed_query(query)
    if faiss is not None:
        normalized_query = query_vector / np.clip(np.linalg.norm(query_vector, axis=1, keepdims=True), 1e-8, None)
        scores, indices = index.search(normalized_query, top_k)
        matches = chunks_df.iloc[indices[0]].copy()
        matches['score'] = scores[0]
        return matches.sort_values('score', ascending=False).reset_index(drop=True)

    distances, indices = index.kneighbors(query_vector, n_neighbors=top_k)
    matches = chunks_df.iloc[indices[0]].copy()
    matches['score'] = 1 - distances[0]
    return matches.sort_values('score', ascending=False).reset_index(drop=True)

def answer_question(query: str) -> str:
    contexts = retrieve(query, top_k=3)
    context_block = '\n\n'.join(f'[{row.source}] {row.text}' for row in contexts.itertuples())
    prompt = f'''You are a financial assistant. Use only the context below when possible.\n\nContext:\n{context_block}\n\nQuestion: {query}\nAnswer with a concise, practical response.'''

    if genai is not None and os.getenv('GOOGLE_API_KEY'):
        try:
            model = genai.GenerativeModel('gemini-1.5-flash')
            response = model.generate_content(prompt)
            return response.text.strip()
        except Exception as exc:
            return f'Gemini generation failed: {exc}\n\nFallback context:\n{context_block}'

    return f'Fallback answer based on retrieved context:\n\n{context_block}'

queries = [
    'How should a diversified portfolio be constructed?',
    'When is sentiment useful in an investment workflow?',
    'What signals support sustainable earnings?',
]

for query in queries:
    print('Q:', query)
    print(answer_question(query))
    print('-' * 80)

Q: How should a diversified portfolio be constructed?
Fallback answer based on retrieved context:

[portfolio_construction] A diversified portfolio should balance expected return, volatility, and correlation across sectors, factors, and capitalizations.

[earnings_quality] Sustainable earnings are supported by margin expansion, recurring revenue, prudent leverage, and positive operating cash flow.

[market_regime] Risk-on regimes generally favor equities, while defensive positioning emphasizes cash flow quality, lower drawdown, and balance sheet strength.
--------------------------------------------------------------------------------
Q: When is sentiment useful in an investment workflow?
Fallback answer based on retrieved context:

[sentiment_signal] Sentiment should be used as a context feature rather than a sole signal. It is most useful when combined with price momentum and liquidity measures.

[earnings_quality] Sustainable earnings are supported by margin expansion, recurring rev

In [5]:
artifact_bundle = {
    'embedding_mode': embedding_mode,
    'embedding_dim': embedding_dim,
    'documents_path': str(corpus_path),
    'chunk_count': int(len(chunks_df)),
}

joblib.dump(vectorizer, MODEL_DIR / 'genai_vectorizer.pkl')
joblib.dump(artifact_bundle, MODEL_DIR / 'genai_metadata.pkl')
joblib.dump(vectorizer, RAG_DIR / 'genai_vectorizer.pkl')
joblib.dump(artifact_bundle, RAG_DIR / 'genai_metadata.pkl')

chunks_df.to_json(MODEL_DIR / 'genai_chunks.json', orient='records', indent=2)
chunks_df.to_json(RAG_DIR / 'genai_chunks.json', orient='records', indent=2)

print('Saved RAG artifacts to:', MODEL_DIR)
print('Saved RAG artifacts to:', RAG_DIR)

Saved RAG artifacts to: F:\project\NEURALALPHA\backend\models\genai
Saved RAG artifacts to: F:\project\NEURALALPHA\ml_pipeline\models\genai


## Backend Integration Notes

The notebook exports retrieval artifacts and metadata only. Any live Gemini access remains optional at inference time and should be configured through environment variables rather than hard-coded in the backend.